# Notebook Databricks — Extração de Dados via Scraping (Bronze / Landing)

**Objetivo:** coletar dados de livros do site público [books.toscrape.com](https://books.toscrape.com/) — um site criado especificamente para prática de web scraping — e armazená-los, em formato bruto (JSON), no **Volume** do Unity Catalog (`lakehouse_catalog.bronze.landing_volume`).

Este notebook utiliza apenas as **funções** definidas em `src/common/scraping_utils.py` e `src/common/spark_utils.py`. Toda a lógica reutilizável fica nos módulos `.py`; a orquestração/execução acontece aqui.

In [ ]:
%pip install requests beautifulsoup4 -q

In [ ]:
import sys, os

# Adiciona a pasta 'src' ao path para importar os módulos de funções
sys.path.append(os.path.abspath("../src"))

from common.spark_utils import volume_path
from common.scraping_utils import scrape_categoria, save_raw_to_volume

In [ ]:
# Categorias do site utilizadas como exemplo de coleta
CATEGORIAS = {
    "Travel": "https://books.toscrape.com/catalogue/category/books/travel_2/index.html",
    "Mystery": "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html",
    "Historical Fiction": "https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html",
}

livros_coletados = []
for categoria, url in CATEGORIAS.items():
    livros_categoria = scrape_categoria(url, categoria, max_paginas=2)
    livros_coletados.extend(livros_categoria)
    print(f"{categoria}: {len(livros_categoria)} livros coletados")

print(f"\nTotal geral: {len(livros_coletados)} livros")

In [ ]:
# Grava o resultado bruto (JSON) no Volume do Unity Catalog
output_dir = volume_path("livros")
os.makedirs(output_dir, exist_ok=True)

caminho_arquivo = save_raw_to_volume(livros_coletados, output_dir)
print(f"Arquivo gravado em: {caminho_arquivo}")

In [ ]:
# Pré-visualização rápida dos dados coletados
import json

with open(caminho_arquivo, encoding="utf-8") as f:
    amostra = json.load(f)[:3]

amostra